# 01 · Integración longitudinal ENASEM

Documento didáctico y reproducible de fuentes, claves, puentes, uniones, titularidad y validaciones. Se reutiliza la base íntegra existente.

Índice: fuentes → identificadores/puentes → uniones → titularidad → validaciones → resultado.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from tesis_enasem.project_paths import project_path, RAW_DIR, PROCESSED_DIR
from tesis_enasem.workspace import display_section, find_variable, load_json
from tesis_enasem.pension_exploration import PopulationSpec
# Configuración de lectura: None muestra toda la evidencia de una sección.
BUSCAR = None
RECALCULAR = False  # No reconstruir bases ni repetir EDA durante la organización.
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


## 1. Fuentes utilizadas

El inventario conserva cada fuente, año verificado, unidad, número de filas, claves y duplicados; 2024 permanece separado y la copia 2018 se utiliza una sola vez.

In [ ]:
inventory = load_json('diccionarios/inventario_fuentes.json')
display(pd.DataFrame(inventory['fuentes']).drop(columns=['sha256'], errors='ignore'))
display_section('fuentes', BUSCAR)

## 2. Identificadores personales

`cunicah` y `np` conservan formatos originales. La función `person_id` concatena con `:` y rechaza componentes ausentes o separadores conflictivos. `ronda` identifica la visita nominal; `a_o_ent` no se sustituye por el año nominal. La misma persona se sigue mediante `id`; la observación única es `id+ronda`. No se imprimen valores individuales.

In [ ]:
from tesis_enasem.integration import person_id, economic_link, left_indexer
import inspect
print(inspect.getsource(person_id))
print(inspect.getsource(economic_link))

## 3. Puentes de 2001 y 2003

Una tabla puente aporta claves económicas que el maestro no contiene de forma equivalente. `sect_k_ind_2001.dta`: maestro→individual por cunicah/codent01→hogar por unhhid. `sect_k_ind_2003.dta`: cunicah/codent03→cunicah/acthog económico. No se utiliza acthog del maestro directamente en 2003. Los duplicados o puentes ambiguos detienen el enlace; no se escoge una fila arbitraria.

In [ ]:
display(pd.DataFrame(inventory['puentes_historicos']))
display_section('integracion', BUSCAR)

## 4. Uniones y cardinalidad

La base maestra se preserva por LEFT JOIN. Fuentes personales: uno-a-uno dentro de ronda; hogar: muchos-a-uno, con claves derechas únicas. Las claves incompletas no se emparejan. El orden original, los tags y todas las variables fuente se conservan; no se suman versiones.

In [ ]:
audit = load_json('docs/integracion/validacion.json')
display(pd.DataFrame(audit['cobertura']))
display(pd.DataFrame(audit['reconciliacion_inventario']))

## 5. Titularidad y múltiples eventos

El enlace de hogar no valida propiedad individual. Recepción propia, rama cónyuge y presencia en hogar permanecen distintas. La orientación exige punteros y filtros documentados; discrepancias conservan desconocido. Viudez corresponde al beneficiario efectivo. Slots, instituciones múltiples y número declarado no son equivalentes.

In [ ]:
ownership = load_json('docs/integracion/titularidad/resultados.json')
display_section('pensiones', BUSCAR)

## 6. Validación real de la base existente

Lectura selectiva de las claves, sin mostrar IDs. No se ejecuta la integración ni se alteran datos.

In [ ]:
from hashlib import sha256
base = PROCESSED_DIR/'enasem_integrada_provisional.parquet'
assert sha256(base.read_bytes()).hexdigest() == audit['hash_parquet']
keys = pd.read_parquet(base, columns=['id','ronda'])
assert len(keys) == audit['N_registros'] == 99676
assert keys['id'].nunique() == audit['N_personas'] == 27171
assert not keys.duplicated(['id','ronda']).any()
display(pd.DataFrame([{'registros':len(keys),'personas':keys.id.nunique(),'duplicados':0,'sha_preservado':True}]))
display_section('validacion', BUSCAR)

## 7. Resultado y reproducción

La base es `data/processed/enasem_integrada_provisional.parquet`; el manifiesto privado describe la versión y la procedencia. `python scripts/run_longitudinal_integration.py` reconstruye el producto solo cuando sea necesario y autorizado; la ejecución ordinaria reutiliza la base. J1/2001, punteros, 33 firmas 2015, no enlaces, pesos/precios y homologaciones pendientes permanecen documentados. No es una clasificación pensionaria definitiva.